In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
df = pd.read_csv("../Data/phishing_features.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [3]:
print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (160064, 13)

First 5 Rows:
                                                                                         url  label  url_length  num_dots  has_https  has_ip  num_subdirs  num_params  suspicious_words  tld  special_char_count  digits_count   entropy
0  http://forum.uk.securebankinggroup.com/107519/8af662/475d964a-0243-46bf-aefd-66da26254efa      1          89         3          0       0            5           0                 2  com                   4            29  4.792985
1                                                                  http://b45042.com/fish/29      1          25         1          0       0            4           0                 0  com                   0             7  4.003856
2                                                             http://bet73018.com/lottery/99      1          30         1          0       0            4           0                 0  com                   0             7  4.053236
3                        

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160064 entries, 0 to 160063
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   url                 160064 non-null  object 
 1   label               160064 non-null  int64  
 2   url_length          160064 non-null  int64  
 3   num_dots            160064 non-null  int64  
 4   has_https           160064 non-null  int64  
 5   has_ip              160064 non-null  int64  
 6   num_subdirs         160064 non-null  int64  
 7   num_params          160064 non-null  int64  
 8   suspicious_words    160064 non-null  int64  
 9   tld                 159243 non-null  object 
 10  special_char_count  160064 non-null  int64  
 11  digits_count        160064 non-null  int64  
 12  entropy             160064 non-null  float64
dtypes: float64(1), int64(10), object(2)
memory usage: 15.9+ MB


In [5]:
display(df.describe(include="all").T)

                       count  unique                top   freq       mean         std       min       25%       50%      75%       max
url                   160064  160064  safeexample84.net      1        NaN         NaN       NaN       NaN       NaN      NaN       NaN
label               160064.0     NaN                NaN    NaN   0.994877    0.071391       0.0       1.0       1.0      1.0       1.0
url_length          160064.0     NaN                NaN    NaN  55.107382  101.782469       6.0      30.0      37.0     65.0   25523.0
num_dots            160064.0     NaN                NaN    NaN   2.998963     1.11921       1.0       2.0       3.0      4.0      59.0
has_https           160064.0     NaN                NaN    NaN   0.434089    0.495638       0.0       0.0       0.0      1.0       1.0
has_ip              160064.0     NaN                NaN    NaN   0.485475    0.499791       0.0       0.0       0.0      1.0       1.0
num_subdirs         160064.0     NaN                NaN

In [6]:
missing_values = df.isnull().sum()

print("Missing Values:")
display(missing_values[missing_values > 0])

Missing Values:
tld    821


In [7]:
print("Class Distribution:")
display(df["label"].value_counts())

print("\nClass Distribution Percentage:")
display(df["label"].value_counts(normalize=True) * 100)

Class Distribution:
label
1    159244
0       820

Class Distribution Percentage:
label
1    99.487705
0     0.512295


In [8]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="label"
)

plt.title("Distribution of Phishing and Legitimate URLs")
plt.xlabel("Label")
plt.ylabel("Number of Samples")

plt.show()

In [9]:
print("Total Rows:", len(df))

print("Duplicate URLs:", df["url"].duplicated().sum())

print("Duplicate Feature Rows:", 
      df.drop(columns=["url"]).duplicated().sum())

Total Rows: 160064
Duplicate URLs: 0
Duplicate Feature Rows: 47348


In [10]:
numeric_columns = df.select_dtypes(include=np.number).columns

df[numeric_columns].hist(
    figsize=(16, 12),
    bins=30
)

plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()

plt.show()

In [11]:
df = df.dropna(subset=["url", "label"])

df = df.drop_duplicates(
    subset=["url"],
    keep="first"
)

df = df.reset_index(drop=True)

print("Dataset Shape After Removing Duplicates:", df.shape)

Dataset Shape After Removing Duplicates: (160064, 13)


In [12]:
feature_columns = [
    "url_length",
    "num_dots",
    "has_https",
    "has_ip",
    "num_subdirs",
    "num_params",
    "suspicious_words",
    "tld",
    "special_char_count",
    "digits_count",
    "entropy"
]

X = df[feature_columns].copy()

y = df["label"].copy()

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

Feature Shape: (160064, 11)
Target Shape: (160064,)

Features:
['url_length', 'num_dots', 'has_https', 'has_ip', 'num_subdirs', 'num_params', 'suspicious_words', 'tld', 'special_char_count', 'digits_count', 'entropy']


In [13]:
print(X.dtypes)

url_length              int64
num_dots                int64
has_https               int64
has_ip                  int64
num_subdirs             int64
num_params              int64
suspicious_words        int64
tld                    object
special_char_count      int64
digits_count            int64
entropy               float64
dtype: object


In [14]:
label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

print("Original Labels:", label_encoder.classes_)
print("Encoded Labels:", np.unique(y))

print("\nLabel Mapping:")

for encoded, original in enumerate(label_encoder.classes_):
    print(f"{encoded} = {original}")

Original Labels: [0 1]
Encoded Labels: [0 1]

Label Mapping:
0 = 0
1 = 1


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (128051, 11)
Testing Shape: (32013, 11)


In [16]:
print("Training Distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting Distribution:")
print(pd.Series(y_test).value_counts())

Training Distribution:
1    127395
0       656
Name: count, dtype: int64

Testing Distribution:
1    31849
0      164
Name: count, dtype: int64


In [17]:
numeric_features = [
    "url_length",
    "num_dots",
    "has_https",
    "has_ip",
    "num_subdirs",
    "num_params",
    "suspicious_words",
    "special_char_count",
    "digits_count",
    "entropy"
]

categorical_features = [
    "tld"
]

In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [19]:
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

In [20]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [21]:
print("Class Distribution Before Undersampling:")
print(pd.Series(y_train).value_counts())

rus = RandomUnderSampler(random_state=42)
X_train, y_train = rus.fit_resample(X_train, y_train)

print("\nClass Distribution After Undersampling:")
print(pd.Series(y_train).value_counts())

Class Distribution Before Undersampling:
1    127395
0       656
Name: count, dtype: int64

Class Distribution After Undersampling:
0    656
1    656
Name: count, dtype: int64


In [22]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    )
}

print("All four models defined successfully!")

All four models defined successfully!


In [23]:
trained_models = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(
        X_train,
        y_train
    )

    trained_models[name] = pipeline

    print(f"{name} trained successfully!")

print("\nAll models trained!")


Training Random Forest...
Random Forest trained successfully!

Training XGBoost...
XGBoost trained successfully!

Training Decision Tree...
Decision Tree trained successfully!

Training Logistic Regression...
Logistic Regression trained successfully!

All models trained!


In [24]:
results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test)
    probabilities = model.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    minority_class = pd.Series(y_test).value_counts().idxmin()
    minority_index = list(model.classes_).index(minority_class)
    minority_probabilities = probabilities[:, minority_index]
    y_minority = (y_test == minority_class).astype(int)

    minority_recall = recall_score(
        y_minority,
        (y_pred == minority_class).astype(int),
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_minority,
        minority_probabilities
    )

    pr_auc = average_precision_score(
        y_minority,
        minority_probabilities
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1,
        "Minority Recall": minority_recall,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by=["Macro F1", "Minority Recall"],
        ascending=[False, False]
    )
)

                 Model  Accuracy  Macro Precision  Macro Recall  Macro F1  Minority Recall  ROC-AUC  PR-AUC
0        Random Forest  1.000000         1.000000      1.000000  1.000000              1.0      1.0     1.0
1              XGBoost  1.000000         1.000000      1.000000  1.000000              1.0      1.0     1.0
2        Decision Tree  1.000000         1.000000      1.000000  1.000000              1.0      1.0     1.0
3  Logistic Regression  0.999844         0.985207      0.999922  0.992453              1.0      1.0     1.0


In [25]:
for name, model in trained_models.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    y_pred = model.predict(X_test)

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1],
            target_names=[
                str(label_encoder.classes_[0]),
                str(label_encoder.classes_[1])
            ],
            zero_division=0
        )
    )


Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       164
           1       1.00      1.00      1.00     31849

    accuracy                           1.00     32013
   macro avg       1.00      1.00      1.00     32013
weighted avg       1.00      1.00      1.00     32013


XGBoost
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       164
           1       1.00      1.00      1.00     31849

    accuracy                           1.00     32013
   macro avg       1.00      1.00      1.00     32013
weighted avg       1.00      1.00      1.00     32013


Decision Tree
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       164
           1       1.00      1.00      1.00     31849

    accuracy                           1.00     32013
   macro avg       1.00      1.00      1.00     32013
weighted avg       1.00      1.00  

In [26]:
for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_encoder.classes_
    )

    disp.plot(cmap="Blues")

    plt.title(f"Confusion Matrix - {name}")
    plt.show()

In [27]:
for name, model in trained_models.items():

    probabilities = model.predict_proba(X_test)

    minority_class = pd.Series(y_test).value_counts().idxmin()

    minority_index = list(model.classes_).index(minority_class)

    minority_probabilities = probabilities[:, minority_index]

    y_minority = (y_test == minority_class).astype(int)

    pr_auc = average_precision_score(
        y_minority,
        minority_probabilities
    )

    roc_auc = roc_auc_score(
        y_minority,
        minority_probabilities
    )

    print(f"\n{name}")
    print("PR-AUC:", pr_auc)
    print("ROC-AUC:", roc_auc)


Random Forest
PR-AUC: 1.0
ROC-AUC: 1.0

XGBoost
PR-AUC: 1.0
ROC-AUC: 1.0

Decision Tree
PR-AUC: 1.0
ROC-AUC: 1.0

Logistic Regression
PR-AUC: 1.0
ROC-AUC: 1.0


In [28]:
best_model_name = results_df.sort_values(
    by=["Macro F1", "Minority Recall"],
    ascending=[False, False]
).iloc[0]["Model"]

best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Random Forest


In [29]:
os.makedirs("../artifacts", exist_ok=True)

joblib.dump(
    best_model,
    "../artifacts/phishing_model.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [30]:
feature_metadata = {
    "features": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

with open(
    "../artifacts/feature_names.json",
    "w"
) as f:

    json.dump(feature_metadata, f, indent=4)

print("Feature metadata saved!")

Feature metadata saved!


In [31]:
model_metadata = {
    "project": "PhishGuard AI",
    "model_name": best_model_name,
    "dataset": "phishing_features.csv",
    "training_samples": int(len(X_train)),
    "testing_samples": int(len(X_test)),
    "features_count": len(feature_columns),
    "imbalance_handling": "Random Undersampling",
    "selection_metric": "Macro F1",
    "label_mapping": {
        str(i): str(label)
        for i, label in enumerate(label_encoder.classes_)
    },
    "metrics": results_df.set_index("Model").loc[
        best_model_name
    ].to_dict()
}

with open(
    "../artifacts/model_metadata.json",
    "w"
) as f:

    json.dump(model_metadata, f, indent=4, default=float)

print("Model metadata saved!")

Model metadata saved!


In [32]:
loaded_model = joblib.load(
    "../artifacts/phishing_model.pkl"
)

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [33]:
sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]

probabilities = loaded_model.predict_proba(sample)[0]

predicted_label = label_encoder.inverse_transform(
    [prediction]
)[0]

print("Predicted Label:", predicted_label)

print("Class Probabilities:")

for class_index, probability in enumerate(probabilities):

    original_label = label_encoder.inverse_transform(
        [class_index]
    )[0]

    print(
        f"Class {original_label}: {probability:.4f}"
    )

Predicted Label: 0
Class Probabilities:
Class 0: 0.9850
Class 1: 0.0150
